In [1]:
# ==============================================================================
# HEMISPHERICAL IMAGE PROCESSING AND CANOPY OPENNESS ESTIMATION
#
# @date Aug 10th, 2026
# @author Diego García Castillo
# ==============================================================================
#
# This script processes hemispherical photographs collected at Atta colony
# sampling sites to quantify local canopy openness.
#
# The workflow includes:
#   1. Loading the original hemispherical photographs.
#   2. Converting each image to grayscale.
#   3. Applying gamma correction according to field-recorded sky conditions.
#   4. Separating illuminated and shaded pixels using image thresholding.
#   5. Applying a predefined circular fisheye mask to restrict the analysis
#      to the standardized region of interest.
#   6. Calculating canopy openness as the percentage of illuminated pixels
#      within the masked region.
#   7. Exporting the calculated values and diagnostic images for
#      reproducibility and visual verification.
#
# Input:
#   - Original hemispherical photographs
#   - Fieldwork metadata
#   - Predefined fisheye image mask (Mask_White_FishEye.jpeg)
#
# Output:
#   - CSV file containing the calculated canopy openness values
#   - PDF report documenting the image-processing results
#
# ==============================================================================

import pandas as pd
import os
import cv2
import numpy as np
from fpdf import FPDF

# --- GLOBAL VARIABLES ---
PHOTOS_PATH = "../data/hemispherical_photographs/" 
DATA_FIELDWORK_CSV = "../data/Ants_Fieldwork_Standard_Data.csv"
OUTPUT_CSV = "../AntsLightProcessed_SI_en.csv"
OUTPUT_PDF = "../ReportLightAnalysisAnts_SI_en.pdf"

# Path of the pre defined mask
PATH_MASK = "../data/" 

# Gamma calibration dictionary
GAMMA_MAP = {'S': 1.0, 'P': 1.5, 'C': 2.0}

# Load the global master mask
img_pared = cv2.imread(os.path.join(PATH_MASK, "Mask_White_FishEye.jpeg"), 0)
_, STANDARD_MASK = cv2.threshold(img_pared, 50, 255, cv2.THRESH_BINARY)

In [2]:
# ==============================================================================
#     Adjust the master fisheye mask to match the orientation and dimensions
#     of a hemispherical photograph.
# 
#     The function first checks whether the photograph and the master mask
#     have different orientations (portrait vs. landscape). If necessary,
#     the mask is rotated 90 degrees clockwise. The mask is then resized to
#     exactly match the dimensions of the photograph.
# 
#     Nearest-neighbor interpolation is used during resizing to preserve the
#     binary structure of the mask.
# ==============================================================================

def adjust_mask_to_photo(img_foto, master_mask):
    h_f, w_f = img_foto.shape
    h_m, w_m = master_mask.shape
    mask_res = master_mask.copy()
    
    # Rotate if necessary
    if (h_f > w_f) != (h_m > w_m):
        mask_res = cv2.rotate(mask_res, cv2.ROTATE_90_CLOCKWISE)
    
    # Resize dimensions if necessary
    if mask_res.shape != (h_f, w_f):
        mask_res = cv2.resize(mask_res, (w_f, h_f), interpolation=cv2.INTER_NEAREST)
    return mask_res

In [3]:
# ==============================================================================
#  Apply gamma correction to adjust image brightness.
# ==============================================================================
def apply_gamma(img, gamma=1.0):
    if gamma == 1.0: return img
    inv_gamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    return cv2.LUT(img, table)

In [7]:
# ==============================================================================
# --- MAIN PROCESSING ---
# ==============================================================================

df = pd.read_csv(DATA_FIELDWORK_CSV)
light_results = []

# PDF configuration
pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()

# ==============================================================================
# 🆕 ADD SCIENTIFIC HEADER (First page only)
# ==============================================================================
pdf.set_margins(left=15, top=15, right=15)

# Supplementary Dataset title
pdf.set_font("Arial", 'B', 12)
pdf.multi_cell(0, 6, "Supplementary Dataset S1. Characterization of canopy openness across sampled colonies via hemispherical photography.", align='L')
pdf.ln(2)

# Main manuscript title (italicized to distinguish it)
pdf.set_font("Arial", 'I', 10)
pdf.multi_cell(0, 5, "Manuscript title: Plasticity vs. specialization: Does niche breadth predict resilience to Anthropogenic land-use change in ecosystem engineers?", align='L')
pdf.ln(2)

# Author and year
pdf.set_font("Arial", '', 10)
pdf.cell(0, 5, "Author: Garcia Castillo, D. 2026.", align='L')

# Divider line
pdf.set_draw_color(180, 180, 180)
pdf.line(15, pdf.get_y() + 4, 195, pdf.get_y() + 4)
pdf.ln(8)
# ==============================================================================

# Variables controlling PDF row layout
current_row = 0
row_distance_between = 85 

# 🆕 Dynamically adjust the initial offset: the first page starts lower 
# to leave room for the header; subsequent pages start at the standard top margin.
y_offset_initial = pdf.get_y() 

for index, row in df.iterrows():
    name_photo = row['Photo_File']
    code_sky = row['Sky']
    value_gamma = GAMMA_MAP.get(code_sky, 1.0)
    
    path_img = os.path.join(PHOTOS_PATH, name_photo)
    if not os.path.exists(path_img):
        print(f"⚠️ File not found: {name_photo}")
        light_results.append(None)
        continue

    # 1. SCIENTIFIC IMAGE LOADING AND PROCESSING (High resolution)
    img_gray = cv2.imread(path_img, 0)

    img_corr = apply_gamma(img_gray, value_gamma)
    
    # Adjust the master mask to the resolution of this image
    mask_adjusted = adjust_mask_to_photo(img_corr, STANDARD_MASK)
    
    # Otsu binarization
    _, binary = cv2.threshold(img_corr, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Apply the mask (this is the final image used for calculations)
    final = cv2.bitwise_and(binary, mask_adjusted)
    
    # 2. CALCULATIONS (Using the original data without loss of precision)
    pixels_light = np.count_nonzero(final)
    area_study = np.count_nonzero(mask_adjusted)
    percentage = round((pixels_light / area_study) * 100, 2)
    light_results.append(percentage)

    # 3. VISUAL PREPARATION FOR THE PDF (File-size optimization)
    img_viz_orig = img_gray.copy()
    img_viz_bin  = final.copy()
    
    # Rotate portrait images so they appear horizontally in the report
    h, w = img_viz_orig.shape
    if h > w:
        img_viz_orig = cv2.rotate(img_viz_orig, cv2.ROTATE_90_CLOCKWISE)
        img_viz_bin  = cv2.rotate(img_viz_bin, cv2.ROTATE_90_CLOCKWISE)

    # IMAGE DOWNSIZING
    max_width_pdf = 800
    ratio = max_width_pdf / float(img_viz_orig.shape[1])
    new_height = int(img_viz_orig.shape[0] * ratio)
    
    img_small_o = cv2.resize(img_viz_orig, (max_width_pdf, new_height), interpolation=cv2.INTER_AREA)
    img_small_b = cv2.resize(img_viz_bin, (max_width_pdf, new_height), interpolation=cv2.INTER_AREA)

    # 4. PDF PAGE MANAGEMENT (3 rows per page)
    if current_row == 3:
        pdf.add_page()
        current_row = 0
        # 🆕 Reset the offset to 20 mm for page 2 onward
        y_offset_initial = 20
    
    y_pos = y_offset_initial + (current_row * row_distance_between)
    
    # Write metadata to the PDF (shifted to x=15 to align with the new margin)
    pdf.set_xy(15, y_pos)
    pdf.set_font("Arial", 'B', 10)
    pdf.cell(0, 10, f"ID: {name_photo} | Sky: {code_sky} (Gamma: {value_gamma}) | Light: {percentage}%", ln=True)
    
    # Save temporary files using 75% JPEG compression
    tmp_o = f"tmp_o_{index}.jpg"
    tmp_b = f"tmp_b_{index}.jpg"
    cv2.imwrite(tmp_o, img_small_o, [int(cv2.IMWRITE_JPEG_QUALITY), 75])
    cv2.imwrite(tmp_b, tmp_b_img := img_small_b, [int(cv2.IMWRITE_JPEG_QUALITY), 75])
    
    # Insert images into the PDF (symmetrically aligned to the 15 mm margins)
    pdf.image(tmp_o, x=15, y=y_pos + 10, w=85, h=65) 
    pdf.image(tmp_b, x=110, y=y_pos + 10, w=85, h=65)
    
    # Delete temporary files
    if os.path.exists(tmp_o): os.remove(tmp_o)
    if os.path.exists(tmp_b): os.remove(tmp_b)
    
    current_row += 1
    print(f"✅ Processed {index+1}/{len(df)}: {name_photo} ({percentage}%)")

✅ Processed 1/117: IMG_20260207_081903.jpg (81.15%)
✅ Processed 2/117: IMG_20260207_083925.jpg (74.15%)
✅ Processed 3/117: IMG_20260207_085621.jpg (66.35%)
✅ Processed 4/117: IMG_20260207_091608.jpg (58.7%)
✅ Processed 5/117: IMG_20260207_093449.jpg (48.3%)
✅ Processed 6/117: IMG_20260207_094951.jpg (45.89%)
✅ Processed 7/117: IMG_20260207_101811.jpg (93.06%)
✅ Processed 8/117: IMG_20260207_104431.jpg (88.27%)
✅ Processed 9/117: IMG_20260207_105957.jpg (82.0%)
✅ Processed 10/117: IMG_20260207_111741.jpg (83.88%)
✅ Processed 11/117: IMG_20260207_115955.jpg (94.9%)
✅ Processed 12/117: IMG_20260207_120311.jpg (92.92%)
✅ Processed 13/117: IMG_20260208_092650.jpg (83.3%)
✅ Processed 14/117: IMG_20260208_094624.jpg (91.65%)
✅ Processed 15/117: IMG_20260208_100201.jpg (91.46%)
✅ Processed 16/117: IMG_20260208_101254.jpg (88.68%)
✅ Processed 17/117: IMG_20260208_104854.jpg (77.68%)
✅ Processed 18/117: IMG_20260208_110517.jpg (72.7%)
✅ Processed 19/117: IMG_20260208_112545.jpg (91.36%)
✅ Proces

In [8]:
# ==============================================================================
# --- SAVE RESULTS AS A CSV AND PDF ---
# ==============================================================================
df['Light_Percentage'] = light_results
df.to_csv(OUTPUT_CSV, index=False)
pdf.output(OUTPUT_PDF)

print(f"\n🚀 Processing complete! CSV saved as '{OUTPUT_CSV}' and PDF saved as '{OUTPUT_PDF}'")


🚀 Processing complete! CSV saved as '/home/diego/AntsLightProcessed_SI_en.csv' and PDF saved as '/home/diego/ReportLightAnalysisAnts_SI_en.pdf'
